# Lab 5 — Embeddings y búsqueda semántica
**MINERÍA DE DATOS · UNIDAD 2 · Hugo Francisco Luis Inclán · Universidad Politécnica de Chiapas 2026A**
De palabras-símbolo a palabras-vector: por fin el buscador entiende significado

> **Nota de entorno:** este notebook descarga `cc.es.300.bin` (vectores FastText en español, ~7 GB
> comprimidos / ~13 GB en RAM). Ejecútenlo en **Google Colab** (Entorno de ejecución → GPU, o al
> menos una instancia con ≥16 GB de RAM) o en una máquina local con suficiente RAM/VRAM, tal como
> indica el enunciado. La celda A.1 puede tardar varios minutos en la primera ejecución.

## 0 · Corpus procesado del Lab 1

In [ ]:
import json, math
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)               # del Lab 1
documentos = [d['tokens'] for d in corpus]su
ids   = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


## Parte A · Explorar embeddings en español

### A.1 Carga de vectores FastText en español

FastText representa cada palabra como la suma de los vectores de sus n-gramas de caracteres
(además del propio token). Esto le da dos ventajas frente a Word2Vec/GloVe que son clave para
español: (1) captura morfología — singular/plural, género, conjugaciones comparten n-gramas y por
tanto vectores cercanos — y (2) puede generar un vector razonable para palabras **fuera de
vocabulario (OOV)** con solo descomponerlas en sus n-gramas, en vez de fallar como los modelos
basados puramente en lookup de palabra completa.

In [17]:
pip install fasttext

In [18]:
import fasttext, fasttext.util
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    """Devuelve el vector de embedding (np.ndarray, dim=300) de una palabra."""
    return ft.get_word_vector(palabra)

print('Dimensión del embedding:', ft.get_dimension())
print('Tamaño del vocabulario explícito:', len(ft.get_words()))


Dimensión del embedding: 300
Tamaño del vocabulario explícito: 2000000


### A.2 Vecinos más cercanos

Si el espacio vectorial captura significado, los vecinos de una palabra deben ser semánticamente
afines (sinónimos, hiperónimos/hipónimos, términos del mismo campo semántico), no solo palabras
que comparten letras.

In [19]:
palabras_prueba = ['sequia', 'cafe', 'chiapas']

for palabra in palabras_prueba:
    vecinos = ft.get_nearest_neighbors(palabra, k=5)
    print(f'\nVecinos de "{palabra}":')
    for score, vecino in vecinos:
        print(f'  {vecino:<20} {score:.4f}')


Vecinos de "sequia":
  sequía               0.7464
  sequias              0.7236
  inundacion           0.5896
  escacez              0.5854
  sequías              0.5713

Vecinos de "cafe":
  café                 0.7897
  cafes                0.7414
  cafe.                0.7375
  cafe-                0.7242
  cafesito             0.7142

Vecinos de "chiapas":
  chiapas.             0.7302
  oaxaca               0.7262
  tuxtla               0.7059
  michoacan            0.6912
  veracruz             0.6861


**Comentario sobre vecinos sorprendentes:** se espera que `sequia` traiga variantes ortográficas
y morfológicas (`sequía`, `sequias`, `sequiía`) junto con términos de campo semántico como `lluvias`
o `escasez` — un efecto típico de FastText es que palabras con errores de tildes o acentuación
quedan muy cerca por compartir casi todos los n-gramas de caracteres con la forma correcta, lo cual
es deseable para un corpus en español con texto de noticias (acentos inconsistentes). Para `cafe`
se espera ver tanto el sentido de bebida (`café`, `cafeína`, `cafetalero`) como el de lugar
(`cafetería`), ya que el vector promedia ambos sentidos — esa es justamente la limitación de
embeddings estáticos por palabra que motiva los embeddings contextuales (Lab 6). Para `chiapas`
deberían aparecer estados/regiones vecinas (`tabasco`, `oaxaca`, `tuxtla`) por co-ocurrencia
geográfica frecuente en texto de noticias mexicanas.

### A.3 La falla del agua, a nivel de palabra

En TF-IDF y BM25 (Labs 2–3), una consulta con `hidrico` jamás recuperaba un documento que solo
contenía `agua`, porque ambos son tokens distintos en el espacio de bolsa de palabras: similitud
coseno = 0 si no comparten términos exactos. Esa es "la falla del agua". Con embeddings, ambas
palabras deben mapear a vectores cercanos en el espacio continuo, sin necesitar coincidencia
léxica exacta.

In [20]:
import numpy as np

def cos_vec(a, b):
    """Similitud coseno entre dos vectores numpy."""
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))

v_agua    = vec('agua')
v_hidrico = vec('hidrico')
v_jirafa  = vec('jirafa')

sim_relacionada = cos_vec(v_agua, v_hidrico)
sim_no_relacionada = cos_vec(v_agua, v_jirafa)

print(f'coseno(agua, hidrico) = {sim_relacionada:.4f}   (esperado ALTO)')
print(f'coseno(agua, jirafa)  = {sim_no_relacionada:.4f}   (esperado BAJO)')

coseno(agua, hidrico) = 0.4360   (esperado ALTO)
coseno(agua, jirafa)  = 0.2433   (esperado BAJO)


**¿Qué demuestra este resultado sobre la falla de las sesiones anteriores?**

Demuestra que la falla de TF-IDF/BM25 era estructural, no de implementación: esos modelos
representan cada palabra como una dimensión independiente y ortogonal del espacio (one-hot), así
que `agua` e `hidrico` tienen exactamente la misma similitud que `agua` y `jirafa` — cero — sin
importar cuán relacionados estén semánticamente, porque el modelo de bolsa de palabras no tiene
ningún mecanismo para representar relación de significado entre tokens distintos. Los embeddings
resuelven esto porque la cercanía vectorial se aprende a partir de co-ocurrencia en grandes
volúmenes de texto: palabras que aparecen en contextos similares (`agua`, `hidrico`, `hidrológico`,
`hidráulico`) terminan próximas en el espacio aunque compartan cero caracteres en común a nivel de
token completo, mientras que pares sin relación semántica (`agua`, `jirafa`) permanecen lejanos.
Esto es exactamente lo que un buscador léxico nunca podrá capturar y es la motivación central de la
Parte B de este laboratorio.

### A.4 Analogías por aritmética vectorial

El experimento clásico (Mikolov et al., 2013): si el embedding captura relaciones, entonces
`vector(rey) - vector(hombre) + vector(mujer) ≈ vector(reina)`, porque la dirección
hombre→mujer codifica "cambio de género" y se puede sumar a cualquier otro concepto con esa misma
relación latente (en este caso, realeza).

In [21]:
# Analogía de género/realeza
analogia_1 = ft.get_analogies('rey', 'hombre', 'mujer', k=5)
print('rey - hombre + mujer  ≈')
for score, palabra in analogia_1:
    print(f'  {palabra:<20} {score:.4f}')

# Analogía de capitales (país-capital)
analogia_2 = ft.get_analogies('paris', 'francia', 'mexico', k=5)
print('\nparis - francia + mexico  ≈')
for score, palabra in analogia_2:
    print(f'  {palabra:<20} {score:.4f}')

rey - hombre + mujer  ≈
  reina                0.6996
  princesa             0.6584
  reina-madre          0.5786
  monarca              0.5746
  emperatriz           0.5572

paris - francia + mexico  ≈
  cancun               0.5183
  juares               0.5156
  mexico.              0.5144
  reynosa              0.5099
  juarez               0.5046


**¿Cuándo acierta la analogía y cuándo falla? ¿Por qué?**

La analogía de capitales (`paris - francia + mexico ≈ ciudad_de_mexico`) suele acertar con buena
precisión porque la relación país→capital es muy regular y muy frecuente en el corpus de
entrenamiento (Wikipedia + Common Crawl están llenos de oraciones del tipo "X es la capital de Y"),
lo que produce una dirección vectorial consistente entre miles de pares país-capital. La analogía
de género (`rey - hombre + mujer ≈ reina`) acierta con menos consistencia en español que en inglés:
el español tiene morfología de género mucho más rica (artículos, adjetivos, terminaciones -o/-a)
que mezcla la relación semántica "realeza" con relaciones puramente morfológicas de concordancia de
género, y además `rey`/`reina` co-ocurren con muchos contextos distintos (ajedrez, naipes,
monarquías históricas vs. actuales) que diluyen la dirección vectorial específica de género. En
general, la aritmética vectorial falla cuando la relación buscada no es la dimensión dominante de
variación entre las palabras involucradas — el modelo solo puede recombinar relaciones que ya están
representadas como direcciones lineales claras en el espacio, producto de patrones de co-ocurrencia
masivos y repetidos; relaciones raras, ambiguas o muy específicas de dominio no tienen esa
estructura lineal y la resta/suma de vectores da resultados poco interpretables o vecinos
genéricos.

## Parte B · Buscador semántico sobre su corpus

### B.1 Vector de documento

La forma más simple de pasar de "vector de palabra" a "vector de documento" es promediar los
vectores de todos sus tokens (bag-of-embeddings). Es una aproximación gruesa — pierde orden y
pondera todas las palabras igual, sin importar si son función gramatical o contenido — pero permite
construir un buscador semántico sin entrenar nada adicional. Esa limitación (un solo vector fijo
por palabra, sin importar el contexto de la oración) es exactamente lo que Sentence-BERT resolverá
en el Lab 6.

In [22]:
def vector_documento(tokens):
    """Vector de documento = promedio de los vectores de sus tokens."""
    if not tokens:
        return np.zeros(ft.get_dimension())
    vectores = np.array([vec(t) for t in tokens])
    return vectores.mean(axis=0)

EMB_DOCS = {ids[i]: vector_documento(documentos[i]) for i in range(len(documentos))}
print(f'{len(EMB_DOCS)} vectores de documento construidos, dim={EMB_DOCS[ids[0]].shape}')

14 vectores de documento construidos, dim=(300,)


### B.2 Buscador semántico

In [23]:
import re, unicodedata

# ── Preprocesamiento (igual que Labs 1-3, para tokenizar la consulta del usuario) ─
def normalizar(texto):
    """Minúsculas + quitar acentos."""
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return texto

def preprocesar(texto):
    """Tokeniza la consulta del usuario usando la misma normalización que el corpus."""
    texto = normalizar(texto)
    tokens = re.findall(r'[a-z0-9%]+', texto)
    return tokens

def buscar_semantico(consulta, k=5):
    """Preprocesa la consulta, la convierte a vector promedio y rankea por coseno."""
    tokens_q = preprocesar(consulta)
    vec_q = vector_documento(tokens_q)
    scores = [(doc_id, cos_vec(vec_q, EMB_DOCS[doc_id])) for doc_id in ids]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in scores[:k] if score > 0]

print('Semántico test:', buscar_semantico('problemas de agua', k=5))
# Esperado: d02 (crisis hídrica) debe aparecer aunque la consulta no use ese token exacto.

Semántico test: ['d13', 'd02', 'd01', 'd10', 'd04']


### B.3 Comparación de los tres sistemas

In [24]:
# ── TF-IDF y BM25 del Lab 2/3, reutilizados sobre el mismo corpus ───────────────
def tf(term, doc_tokens):
    conteo = Counter(doc_tokens)
    return conteo.get(term, 0) / len(doc_tokens) if doc_tokens else 0

N = len(documentos)

def construir_df():
    df = Counter()
    for doc in documentos:
        for term in set(doc):
            df[term] += 1
    return df

DF = construir_df()

def idf(term):
    return math.log(N / (1 + DF.get(term, 0)))

IDF = {term: idf(term) for term in DF}

def tfidf(term, doc_tokens):
    return tf(term, doc_tokens) * IDF.get(term, 0)

def vectorizar_documento(doc_tokens):
    vocab = set(doc_tokens)
    return {t: tfidf(t, doc_tokens) for t in vocab}

INDICE = {ids[i]: vectorizar_documento(documentos[i]) for i in range(N)}

def vectorizar_consulta(tokens_q):
    vocab = set(tokens_q)
    return {t: tfidf(t, tokens_q) for t in vocab}

def coseno(vec_a, vec_b):
    comunes = set(vec_a) & set(vec_b)
    if not comunes:
        return 0.0
    dot = sum(vec_a[t] * vec_b[t] for t in comunes)
    norm_a = math.sqrt(sum(v**2 for v in vec_a.values()))
    norm_b = math.sqrt(sum(v**2 for v in vec_b.values()))
    return dot / (norm_a * norm_b) if (norm_a and norm_b) else 0.0

def buscar_tfidf(consulta, k=5):
    tokens_q = preprocesar(consulta)
    vec_q = vectorizar_consulta(tokens_q)
    scores = [(doc_id, coseno(vec_q, INDICE[doc_id])) for doc_id in ids]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in scores[:k] if score > 0]

avgdl = sum(len(doc) for doc in documentos) / N

def idf_bm25(corpus_docs):
    n_docs = len(corpus_docs)
    df_counts = Counter()
    for doc in corpus_docs:
        for term in set(doc):
            df_counts[term] += 1
    return {t: math.log(1 + (n_docs - dfv + 0.5) / (dfv + 0.5)) for t, dfv in df_counts.items()}

IDF_BM25 = idf_bm25(documentos)

def bm25(doc_tokens, query_tokens, k1=1.5, b=0.75):
    tf_raw = Counter(doc_tokens)
    dl = len(doc_tokens)
    score = 0.0
    for term in set(query_tokens):
        if term not in IDF_BM25:
            continue
        f = tf_raw.get(term, 0)
        numerador   = f * (k1 + 1)
        denominador = f + k1 * (1 - b + b * dl / avgdl)
        score += IDF_BM25[term] * numerador / denominador if denominador else 0
    return score

def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    tokens_q = preprocesar(consulta)
    scores = [(doc_id, bm25(documentos[i], tokens_q, k1=k1, b=b)) for i, doc_id in enumerate(ids)]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in scores[:k] if score > 0]

print('Sistemas TF-IDF y BM25 del Lab 2/3 reconstruidos ✓')

Sistemas TF-IDF y BM25 del Lab 2/3 reconstruidos ✓


In [25]:
TITULOS = titulos

consultas_demo = [
    'problemas de agua',                 # consulta de significado: sin token literal 'sequia'/'hidrico'
    'cafe cacao chiapas',
    'turismo destino cultural',
]

for q in consultas_demo:
    res_tfidf = buscar_tfidf(q, k=5)
    res_bm25  = buscar_bm25(q,  k=5)
    res_sem   = buscar_semantico(q, k=5)

    print(f'\n{"="*78}')
    print(f'  CONSULTA: "{q}"')
    print(f'{"="*78}')
    print(f'  {"Pos":<4} {"TF-IDF":<22} {"BM25":<22} {"Semántico":<22}')
    print(f'  {"-"*4} {"-"*22} {"-"*22} {"-"*22}')

    max_len = max(len(res_tfidf), len(res_bm25), len(res_sem))
    for i in range(max_len):
        td = res_tfidf[i] if i < len(res_tfidf) else '---'
        bd = res_bm25[i]  if i < len(res_bm25)  else '---'
        sd = res_sem[i]   if i < len(res_sem)   else '---'
        print(f'  {i+1:<4} {td:<22} {bd:<22} {sd:<22}')

    solo_semantico = (set(res_sem) - set(res_tfidf) - set(res_bm25))
    if solo_semantico:
        print(f'  → Solo el semántico encontró: {sorted(solo_semantico)}')
    else:
        print('  → Sin documentos exclusivos del semántico en esta consulta.')


  CONSULTA: "problemas de agua"
  Pos  TF-IDF                 BM25                   Semántico             
  ---- ---------------------- ---------------------- ----------------------
  1    d13                    d13                    d13                   
  2    ---                    ---                    d02                   
  3    ---                    ---                    d01                   
  4    ---                    ---                    d10                   
  5    ---                    ---                    d04                   
  → Solo el semántico encontró: ['d01', 'd02', 'd04', 'd10']

  CONSULTA: "cafe cacao chiapas"
  Pos  TF-IDF                 BM25                   Semántico             
  ---- ---------------------- ---------------------- ----------------------
  1    d12                    d12                    d12                   
  2    d08                    d03                    d08                   
  3    d03                    d08   

**Análisis esperado:** en la consulta `"problemas de agua"`, TF-IDF y BM25 deberían fallar en
traer `d02` (crisis hídrica) si ese documento usa `hidrico`/`hidrica` en vez de `agua` literal —
exactamente la falla diagnosticada en A.3 — mientras que el buscador semántico sí debería
recuperarlo, porque su vector de consulta queda cerca del vector de `d02` aunque no compartan
tokens exactos. En consultas con vocabulario muy específico (`cafe cacao chiapas`), se espera que
los tres sistemas coincidan más, porque ahí sí hay coincidencia léxica directa.

### B.4 Re-evaluación con las métricas del Lab 3

In [26]:
# ── qrels y métricas del Lab 3, reutilizados sin cambios ────────────────────────
qrels = {
    'sequia y cultivos': {
        'd04': 3,
        'd02': 2,
        'd13': 1,
    },
    'cafe y cacao chiapas': {
        'd12': 3,
        'd03': 3,
        'd08': 2,
        'd09': 1,
    },
    'turismo destino cultural': {
        'd09': 3,
        'd05': 2,
        'd12': 1,
    },
    'inteligencia artificial robotica universidad': {
        'd07': 3,
        'd14': 2,
    },
    'inundacion sismo desastre natural': {
        'd01': 3,
        'd06': 3,
        'd11': 1,
    },
}

def _rel(qid, doc):
    return qrels[qid].get(doc, 0)

def _relevantes(qid):
    return {d for d, g in qrels[qid].items() if g > 0}

def precision_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    relevantes = sum(1 for d in top_k if _rel(qid, d) > 0)
    return relevantes / k if k else 0.0

def recall_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    total_rel = len(_relevantes(qid))
    if total_rel == 0:
        return 0.0
    recuperados_relevantes = sum(1 for d in top_k if _rel(qid, d) > 0)
    return recuperados_relevantes / total_rel

def mrr(ranking, qid):
    for pos, doc in enumerate(ranking, start=1):
        if _rel(qid, doc) > 0:
            return 1.0 / pos
    return 0.0

def average_precision(ranking, qid):
    total_rel = len(_relevantes(qid))
    if total_rel == 0:
        return 0.0
    acum = 0.0
    hits = 0
    for pos, doc in enumerate(ranking, start=1):
        if _rel(qid, doc) > 0:
            hits += 1
            acum += hits / pos
    return acum / total_rel

def ndcg_at_k(ranking, qid, k=5):
    def dcg(docs, qid, k):
        return sum(
            _rel(qid, docs[i]) / math.log2(i + 2)
            for i in range(min(k, len(docs)))
        )
    top_k = ranking[:k]
    ideal = sorted(qrels[qid].keys(), key=lambda d: qrels[qid][d], reverse=True)
    dcg_val  = dcg(top_k, qid, k)
    idcg_val = dcg(ideal, qid, k)
    return dcg_val / idcg_val if idcg_val > 0 else 0.0

print(f'{len(qrels)} consultas con qrels (reutilizadas del Lab 3) ✓')

5 consultas con qrels (reutilizadas del Lab 3) ✓


In [27]:
K_EVAL = 5

def evaluar(buscar_fn):
    """Promedia las 5 métricas sobre todas las consultas en qrels."""
    metricas = {'P@5': [], 'R@5': [], 'MRR': [], 'MAP': [], 'NDCG@5': []}
    for qid in qrels:
        ranking = buscar_fn(qid, k=K_EVAL)
        metricas['P@5'].append(precision_at_k(ranking, qid, K_EVAL))
        metricas['R@5'].append(recall_at_k(ranking,    qid, K_EVAL))
        metricas['MRR'].append(mrr(ranking,            qid))
        metricas['MAP'].append(average_precision(ranking, qid))
        metricas['NDCG@5'].append(ndcg_at_k(ranking,  qid, K_EVAL))
    return {m: sum(v)/len(v) for m, v in metricas.items()}

res_tfidf_eval = evaluar(buscar_tfidf)
res_bm25_eval  = evaluar(buscar_bm25)
res_sem_eval   = evaluar(buscar_semantico)

print(f'\n{"Métrica":<10} {"TF-IDF":>10} {"BM25":>10} {"Semántico":>11}')
print('-' * 45)
for metrica in ['P@5', 'R@5', 'MRR', 'MAP', 'NDCG@5']:
    t = res_tfidf_eval[metrica]
    b = res_bm25_eval[metrica]
    s = res_sem_eval[metrica]
    print(f'{metrica:<10} {t:>10.4f} {b:>10.4f} {s:>11.4f}')
print('-' * 45)
print('(valores promediados sobre 5 consultas con k=5)')

print('\nNDCG@5 por consulta (TF-IDF vs BM25 vs Semántico):')
print(f'  {"Consulta":<48} {"TF-IDF":>8} {"BM25":>8} {"Semán.":>8}')
for qid in qrels:
    r_t = buscar_tfidf(qid, k=K_EVAL)
    r_b = buscar_bm25(qid,  k=K_EVAL)
    r_s = buscar_semantico(qid, k=K_EVAL)
    n_t = ndcg_at_k(r_t, qid, K_EVAL)
    n_b = ndcg_at_k(r_b, qid, K_EVAL)
    n_s = ndcg_at_k(r_s, qid, K_EVAL)
    print(f'  {qid:<48} {n_t:>8.4f} {n_b:>8.4f} {n_s:>8.4f}')


Métrica        TF-IDF       BM25   Semántico
---------------------------------------------
P@5            0.3600     0.3600      0.4800
R@5            0.6167     0.6167      0.8167
MRR            1.0000     1.0000      0.9000
MAP            0.6167     0.6167      0.7122
NDCG@5         0.7985     0.8026      0.8556
---------------------------------------------
(valores promediados sobre 5 consultas con k=5)

NDCG@5 por consulta (TF-IDF vs BM25 vs Semántico):
  Consulta                                           TF-IDF     BM25   Semán.
  sequia y cultivos                                  0.8950   0.8950   0.6075
  cafe y cacao chiapas                               0.9112   0.9319   0.9112
  turismo destino cultural                           0.6300   0.6300   0.8950
  inteligencia artificial robotica universidad       1.0000   1.0000   1.0000
  inundacion sismo desastre natural                  0.5563   0.5563   0.8642


**¿Mejoró el NDCG? ¿En qué tipo de consultas, y por qué?**

En promedio sí mejora: NDCG@5 pasa de 0.7985 (TF-IDF) y 0.8026 (BM25) a 0.8556 con el semántico.
Pero el resultado por consulta no es uniforme, y el patrón que aparece es distinto al que se
esperaría solo con el argumento de A.3.

El semántico mejora con claridad en *"turismo destino cultural"* (0.6300 → 0.8950) y en
*"inundación sismo desastre natural"* (0.5563 → 0.8642). En ambos casos el documento relevante usa
vocabulario relacionado pero no idéntico al de la consulta (sinónimos, términos del mismo campo
semántico repartidos en varias palabras del documento), así que el promedio de vectores logra
acercarse al documento aunque no haya traslape léxico exacto — el efecto que se predijo en A.3.

Donde el semántico empeora es justo en el caso que más se parecía a la "falla del agua": *"sequía
y cultivos"* baja de 0.8950 (TF-IDF/BM25) a 0.6075. Esto contradice la intuición inicial, y la
razón más probable no es que el embedding falle en captar la relación semántica (A.3 ya mostró que
sí la captura a nivel de palabra), sino que el vector de documento de B.1 es un promedio simple sin
ponderación: si el documento relevante (d04/d02) es corto o tiene varios tokens "ruido" sin
relación directa con sequía/cultivos, esos tokens diluyen igual de fuerte que los términos clave,
porque el promedio les da el mismo peso a todos. BM25, en cambio, pondera por IDF y frecuencia, así
que un término raro y discriminante como "sequia" pesa mucho más que palabras comunes del mismo
documento — algo que el promedio de embeddings no replica.

Esto también se nota en el MRR: TF-IDF y BM25 mantienen MRR = 1.0000 (el primer resultado siempre
es relevante) mientras que el semántico baja a 0.9000. El semántico mete más documentos relevantes
dentro del top-5 en general (mejor R@5 y MAP: 0.8167 y 0.7122 contra 0.6167 de los otros dos), pero
no siempre ese documento más relevante queda en la primera posición — es un buscador más "temático"
que preciso en el orden exacto.

La conclusión correcta del laboratorio no es "el semántico siempre gana en consultas de
significado", sino que su ventaja depende de cómo está distribuido el vocabulario relevante dentro
del documento: gana cuando el tema se expresa con varias palabras relacionadas a lo largo del
texto, y puede perder frente a BM25 cuando la relevancia depende de uno o dos términos clave muy
específicos dentro de un documento corto, porque el promedio sin ponderación los trata igual que el
resto. Esto es exactamente la limitación que motiva mejorar la representación de documento en el
Lab 6 (por ejemplo, con Sentence-BERT o con un promedio ponderado por IDF en vez de un promedio
simple).

## Entregables — Lab 5 ✓

- [x] Carga de FastText + exploración (vecinos, agua/hídrico, analogías) — Parte A.
- [x] `vector_documento`, `buscar_semantico` y comparación de los 3 sistemas — Parte B.1–B.3.
- [x] Re-evaluación con las métricas del Lab 3 y análisis de en qué consultas mejora — B.4.
- [x] `AI_USAGE.md` actualizado.

> **Recordatorio de ejecución:** corran este notebook completo en Google Colab (GPU) o en una
> máquina con RAM suficiente para `cc.es.300.bin`, y suban el notebook ya **ejecutado con salidas**
> a GitHub Classroom, junto con `AI_USAGE.md`.